# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [15]:
base_url = "https://shekharx.in"
endpoints=["/","/projects","/blogs"]
links=[item for endpoint in endpoints for item in fetch_website_links(f"{base_url}/{endpoint}")]
links

['mailto:contact.sidshekhar@gmail.com',
 'https://www.linkedin.com/in/shekharsid/',
 'https://github.com/SXsid',
 'https://x.com/shekharTwts',
 'https://github.com/SXsid',
 'https://www.linkedin.com/in/shekharsid/',
 'https://x.com/shekharTwts',
 'https://dev.to/sid04',
 'mailto:contact.sidshekhar@gmail.com',
 '/',
 'https://github.com/SXsid/KitsuDB',
 'https://github.com/SXsid/Rootless-Raft',
 'https://moneybuddy.shekharx.in/dashbord',
 'https://github.com/SXsid/MoneyBuddy',
 'https://github.com/SXsid/KitsuDB',
 'https://github.com/SXsid/Rootless-Raft',
 'https://moneybuddy.shekharx.in/dashbord',
 'https://onlinepal.shekharx.in/',
 'https://github.com/SXsid/Online-Pal',
 'https://carfterai.shekharx.in/',
 'https://github.com/SXsid/ArticleCrafter',
 'https://github.com/SXsid/Mirrorly',
 'https://github.com/SXsid/gomon',
 'https://github.com/SXsid/glsp',
 'https://onlinepal.shekharx.in/',
 'https://github.com/SXsid/Online-Pal',
 'https://carfterai.shekharx.in/',
 'https://github.com/SXs

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [16]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [18]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [19]:
print(get_links_user_prompt("https://shekharx.in"))


Here is the list of links on the website https://shekharx.in -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

mailto:contact.sidshekhar@gmail.com
https://www.linkedin.com/in/shekharsid/
https://github.com/SXsid
https://x.com/shekharTwts
https://github.com/SXsid
https://www.linkedin.com/in/shekharsid/
https://x.com/shekharTwts
https://dev.to/sid04
mailto:contact.sidshekhar@gmail.com


In [20]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [21]:
select_relevant_links("https://shekharx.in")

{'links': [{'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/shekharsid/'},
  {'type': 'github profile', 'url': 'https://github.com/SXsid'},
  {'type': 'x profile', 'url': 'https://x.com/shekharTwts'},
  {'type': 'dev.to profile', 'url': 'https://dev.to/sid04'}]}

In [22]:
# it remove the mail which was not link 
# and it also delt with the dublicte link problem

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [23]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [24]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
yuxinlu1/gemma-4-12B-coder-fable5-composer2.5-v1-GGUF
Updated
2 days ago
•
359k
•
2.04k
zai-org/GLM-5.2
Updated
2 days ago
•
27.4k
•
1.77k
MiniMaxAI/MiniMax-M3
Updated
5 days ago
•
104k
•
1.17k
WeiboAI/VibeThinker-3B
Updated
2 days ago
•
20.3k
•
541
moonshotai/Kimi-K2.7-Code
Updated
6 days ago
•
363k
•
941
Browse 2M+ models
Spaces
Running
on
Zero
Agents
247
LTX 2.3 Finetuned I2V
🎬
247
LTX 

In [25]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [27]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [28]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nyuxinlu1/gemma-4-12B-coder-fable5-composer2.5-v1-GGUF\nUpdated\n2 days ago\n•\n359k\n•\n2.04k\nzai-org/GLM-5.2\nUpdated\n2 day

In [29]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [32]:
create_brochure("Sudhanshu Shekhar", "https://shekharx.in")

# Sudhanshu Shekhar: Backend & Infrastructure Engineering Expert

---

## About Sudhanshu Shekhar

Sudhanshu Shekhar is a passionate self-taught Backend and Infrastructure Engineer based in Delhi, India. With a deep curiosity about building reliable, scalable systems, Sudhanshu specializes in backend systems, asynchronous APIs, and infrastructure monitoring that ensures uptime and stability. He focuses on the fundamentals that keep production systems running smoothly and efficiently, valuing reliability and observability deeply.

---

## Expertise & Technical Skills

### Programming Languages  
- Go  
- TypeScript  
- Python  
- C  
- SQL  

### Backend Frameworks & Technologies  
- Express.js  
- FastAPI  
- Go’s net/http package  
- Langchain  
- gRPC  
- ElasticSearch  

### Databases  
- PostgreSQL  
- Redis  
- Firestore  

### DevOps & Cloud Infrastructure  
- Docker  
- Amazon Web Services (AWS)  
- CI/CD pipelines  
- Monitoring: Grafana & Prometheus  

### Frontend  
- React  
- Next.js  
- Tailwind CSS  
- Redux  
- Tanstack Query  

### Tools  
- Git  
- Vim  
- Claude (AI tools integration)  

---

## Professional Focus

- Designing and shipping robust backend systems  
- Building asynchronous APIs that handle real-world traffic  
- Crafting and maintaining monitoring stacks to prevent downtimes  
- Emphasizing “boring” yet crucial aspects like system reliability and observability  
- Developing developer tools (e.g., *gomon*, a Go-based nodemon alternative widely adopted in the community)  
- Deep dives into system design, database internals, and production-grade backend patterns  

---

## Projects & Contributions

Sudhanshu actively builds open-source tools to fill gaps he encounters in his work, such as **gomon**, which has gained community usage and appreciation. He shares insights via blogging and contributing to tech communities, reflecting his ongoing learning and teaching spirit.

---

## Career & Education

- Bachelor of Science student at IIT Madras  
- Focus on backend development and DevOps  
- Always open to challenging projects that push the boundaries of backend engineering and infrastructure reliability  

---

## Company Culture & Values

Sudhanshu embodies a culture of continuous learning, self-improvement, and pragmatism. He values:

- **Curiosity**: Passion for understanding underlying systems  
- **Reliability**: Prioritizing systems that "just work" without unnecessary firefighting  
- **Simplicity**: Building tools that developers genuinely find useful  
- **Open Source & Community**: Sharing knowledge and tools openly  

---

## How to Connect

- **Email**: contact.sidshekhar@gmail.com  
- **GitHub**: [SXsid - Sudhanshu Shekhar](https://github.com/SXsid)  
- **Twitter (X)**: [@shekharTwts](https://twitter.com/shehkarTwts)  
- **Blog & Projects**: Accessible on personal site shekharx.in  

---

## Why Work With Sudhanshu?

- Proven ability to develop backend systems for production with real traffic  
- Expertise in infrastructure resiliency, observability, and asynchronous system design  
- Focus on shipping quality work that minimizes operational burden (e.g., no 3 AM wake-ups)  
- Passionate about making developer experience better through tooling and automation  
- Collaborative mindset, continuously learning and sharing knowledge  

---

**© 2025 Sudhanshu Shekhar. All rights reserved.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [33]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [34]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Company Brochure

---

## About Hugging Face

**Hugging Face** is the pioneering AI community that is building the future of machine learning. As a collaborative platform, it empowers the global machine learning ecosystem by enabling the creation, discovery, and sharing of models, datasets, and applications. Hugging Face serves as the premier hub where data scientists, researchers, developers, and enterprises come together to innovate and accelerate AI development.

---

## What We Offer

- **Model Hub**: Access over 2 million machine learning models for various tasks, regularly updated and maintained by a vibrant community.
- **Datasets**: Explore and share over 500,000 datasets tailored for machine learning projects across multiple domains.
- **Spaces**: Host and deploy machine learning applications in user-friendly interactive environments powered by state-of-the-art models.
- **Buckets**: Secure storage solutions supporting enterprise and individual needs for managing large amounts of ML data.
- **HuggingChat**: Cutting-edge conversational AI tools designed for seamless, natural language interactions.
- **Enterprise Solutions**: Customized professional support, inference endpoints, and infrastructure tailored to enterprise requirements.
- **Hugging Face PRO**: Enhanced features and support for pro users seeking advanced capabilities.

---

## Our Customers

Hugging Face serves a diverse global clientele including:

- **Researchers and Academics** seeking open access to leading AI models and datasets.
- **Startups and Developers** building innovative AI-powered applications with no heavy infrastructure overhead.
- **Enterprises and Organizations** looking for scalable and secure AI solutions with professional support.
- **AI Enthusiasts and Community Members** exploring daily papers, engaging on forums and Discord, and collaborating via GitHub.

The platform is a cornerstone for anyone involved in machine learning who values open collaboration, cutting-edge technology, and an inclusive ecosystem.

---

## Company Culture

Hugging Face is driven by a spirit of open innovation, inclusivity, and community-driven progress. The company thrives on:

- **Collaboration**: Building a thriving machine learning community that shares knowledge, skills, and resources.
- **Transparency**: Open-source ethos fueling trust, rapid innovation, and accessibility.
- **Passion for AI**: A strong commitment to advancing AI technology responsibly and making it accessible for all.
- **Diversity & Inclusion**: Encouraging contributors worldwide from all backgrounds to participate and share in the AI revolution.
- **Continuous Learning**: Providing resources such as blogs, daily papers, forums, and live chats to foster ongoing education and development.

---

## Careers & Opportunities

Join Hugging Face and be part of the future of AI!

- Work with cutting-edge technologies in machine learning and artificial intelligence.
- Collaborate with a vibrant global community of AI researchers, engineers, and enthusiasts.
- Contribute to open-source projects that power thousands of applications worldwide.
- Enjoy a dynamic, mission-driven culture focused on innovation, learning, and impact.

Check Hugging Face’s website for current job openings in engineering, research, community management, and enterprise support.

---

## Get Involved

- Explore and contribute to over 2 million models and 500,000 datasets.
- Participate in community discussions via Discord, forums, GitHub.
- Use Spaces to deploy AI-driven applications easily.
- Access enterprise-grade AI solutions with dedicated support.
- Stay informed with daily papers, blog posts, and learning resources.

Visit [huggingface.co](https://huggingface.co) and become part of **The AI Community Building the Future** today!

---

### Brand Identity

- Signature colors: bright yellow (#FFD21E), vivid orange (#FF9D00), and neutral gray (#6B7280).
- Official logos and assets are available for collaboration and projects via the Hugging Face brand toolkit.

---

**Hugging Face** — Empowering the world to build smarter AI together.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>